# Graph Feature Extraction — Teacher Soft Labels

This notebook builds molecular graph dictionaries with ground-truth `y` and a five-column `y_soft` tensor. It reads soft-label columns in the **explicit, unchanged order** `temp`, `elem`, `mol`, `SMILES_Kmeans`, `Bemis_Murcko`; check that these names, order, and prediction scales match the teacher-generation and student-training steps.

**Input:** `data-set/input/with_soft_labels.csv` (set the relative path below to your actual input file). **Outputs:** `molecule_graph_data.pt`, `feature_scalers.pkl`, and `graph_metadata.txt` under `data-set/features/soft_labels/`. The original file names and dictionary keys remain unchanged.

Fit normalization only on the intended training data. The current notebook fits new scalers to its whole input; do not independently refit them on a held-out partition. It does not rescale teacher predictions; `y_soft` retains the numeric values from the CSV.

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
import pickle

from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

# --- Feature encoding (original feature definitions and dimensions) ---
def one_of_k_encoding_unk(x, allowable_set):
    """One-hot encoding; map unsupported values to the final category."""
    if x not in allowable_set:
        x = allowable_set[-1]
    return [x == s for s in allowable_set]

def extract_atom_features(atom):
    """Return the original atom feature vector without changing its feature order."""
    atom_idx = int(atom.GetIdx())
    allowable_elements = ['C','N','O','S','F','H','Si','P','Cl','Br',
                          'Li','Na','K','Mg','Ca','Fe','As','Al','I','B',
                          'V','Tl','Sb','Sn','Ag','Pd','Co','Se','Ti','Zn',
                          'Ge','Cu','Au','Ni','Cd','Mn','Cr','Pt','Hg','Pb']
    
    basic_features = one_of_k_encoding_unk(atom.GetSymbol(), allowable_elements) + \
                     one_of_k_encoding_unk(atom.GetDegree(), [0,1,2,3,4,5]) + \
                     one_of_k_encoding_unk(atom.GetTotalNumHs(), [0,1,2,3,4]) + \
                     one_of_k_encoding_unk(atom.GetImplicitValence(), [0,1,2,3,4,5]) + \
                     [atom.GetIsAromatic()]
    
    extra_features = [
        atom.GetAtomicNum() / 100.0,
        atom.GetMass() / 100.0,
        atom.GetFormalCharge(),
        atom.GetNumRadicalElectrons(),
        atom.GetNumExplicitHs(),
    ]
    
    if atom.HasProp('_GasteigerCharge'):
        extra_features.append(float(atom.GetProp('_GasteigerCharge')))
    else:
        extra_features.append(0.0)
    
    extra_features += one_of_k_encoding_unk(atom.GetHybridization(), list(Chem.rdchem.HybridizationType.values))
    extra_features += one_of_k_encoding_unk(atom.GetChiralTag(), list(Chem.rdchem.ChiralType.values))
    
    ring_features = [atom.IsInRing()]
    for ring_size in range(3, 8):
        ring_features.append(atom.IsInRingSize(ring_size))
    extra_features += ring_features
    
    return np.array(basic_features + extra_features, dtype=float)

def extract_bond_features(bond):
    """Encode bond type, ring membership, conjugation, and stereochemistry."""
    bond_type = bond.GetBondType()
    bond_features = [
        bond_type == Chem.rdchem.BondType.SINGLE,
        bond_type == Chem.rdchem.BondType.DOUBLE,
        bond_type == Chem.rdchem.BondType.TRIPLE,
        bond_type == Chem.rdchem.BondType.AROMATIC,
        bond.IsInRing(),
        bond.GetIsConjugated(),
        bond.GetStereo() != Chem.rdchem.BondStereo.STEREONONE
    ]
    return np.array(bond_features, dtype=float)

def extract_molecule_global_features(mol):
    """Combine eight molecular descriptors with a 64-bit Morgan fingerprint."""
    physicochem_features = [
        Descriptors.MolWt(mol),
        Descriptors.MolLogP(mol),
        Descriptors.TPSA(mol),
        Descriptors.NumHAcceptors(mol),
        Descriptors.NumHDonors(mol),
        rdMolDescriptors.CalcNumRotatableBonds(mol),
        rdMolDescriptors.CalcNumHeteroatoms(mol),
        rdMolDescriptors.CalcFractionCSP3(mol),
    ]
    
    morgan_fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=64)
    fp_features = list(morgan_fp)
    
    return physicochem_features + fp_features

class MoleculeGraphBuilder:
    """Build padded graph dictionaries with the original feature schema."""
    def __init__(self, atom_scaler=None, bond_scaler=None):
        self.atom_scaler = atom_scaler
        self.bond_scaler = bond_scaler
    
    def build_graph_from_smiles(self, smiles, max_atoms, use_bond_features=True, use_global_features=True):
        """Return graph data or None if the provided SMILES is invalid."""
        parsed_mol = Chem.MolFromSmiles(smiles) if isinstance(smiles, str) else None
        mol = Chem.AddHs(parsed_mol) if parsed_mol is not None else None
        if mol is None: 
            return None
        
        atom_features_list = [extract_atom_features(atom) for atom in mol.GetAtoms()]
        num_atoms = len(atom_features_list)
        atom_feature_dim = len(atom_features_list[0]) if num_atoms > 0 else 0
        
        padded_atom_features = np.zeros((max_atoms, atom_feature_dim), dtype=float)
        if num_atoms > 0:
            padded_atom_features[:num_atoms] = np.vstack(atom_features_list)
            if self.atom_scaler is not None:
                padded_atom_features = self.atom_scaler.transform(padded_atom_features)
        
        edge_indices = []
        bond_features_list = []
        
        for bond in mol.GetBonds():
            atom_i = int(bond.GetBeginAtomIdx())
            atom_j = int(bond.GetEndAtomIdx())
            edge_indices.append([atom_i, atom_j])
            edge_indices.append([atom_j, atom_i])
            
            if use_bond_features:
                bond_feat = extract_bond_features(bond)
                bond_features_list.append(bond_feat)
                bond_features_list.append(bond_feat)
        
        for atom_idx in range(num_atoms):
            edge_indices.append([atom_idx, atom_idx])
            if use_bond_features:
                bond_feat_dim = len(bond_features_list[0]) if bond_features_list else 7
                self_loop_bond_feat = np.zeros(bond_feat_dim)
                bond_features_list.append(self_loop_bond_feat)
        
        edge_index_tensor = torch.tensor(edge_indices).t().contiguous()
        
        bond_attr_tensor = None
        if use_bond_features and len(bond_features_list) > 0:
            bond_attr_tensor = torch.tensor(np.asarray(bond_features_list), dtype=torch.float)
            if self.bond_scaler is not None:
                bond_attr_tensor = torch.tensor(self.bond_scaler.transform(bond_attr_tensor.numpy()), dtype=torch.float)
        
        global_features_tensor = None
        if use_global_features:
            try:
                global_features = extract_molecule_global_features(mol)
                global_features_tensor = torch.tensor(global_features, dtype=torch.float).view(1, -1)
            except Exception as e:
                global_features_tensor = None
        
        return {
            'x': torch.tensor(padded_atom_features, dtype=torch.float),
            'edge_index': edge_index_tensor,
            'edge_attr': bond_attr_tensor,
            'u': global_features_tensor,
            'smiles': smiles,
            'num_atoms': num_atoms
        }


# --- Graph construction and artifact export ---
def extract_molecule_graph_features(
    df,
    soft_label_columns,
    output_directory="molecule_graph_features",
    max_atoms=None,
    use_bond_features=True,
    use_global_features=True
):
    """Fit feature scalers and serialize graphs with teacher soft labels."""
    if not soft_label_columns:
        raise ValueError("At least one soft-label column is required.")
    required_columns = ["SMILES", "y_true", *soft_label_columns]
    missing = [column for column in required_columns if column not in df.columns]
    if missing:
        raise ValueError(f"Missing required input columns: {missing}")
    if df.empty:
        raise ValueError("The input CSV contains no rows.")
    os.makedirs(output_directory, exist_ok=True)
    
    if max_atoms is None:
        atom_counts = []
        for smiles in df['SMILES']:
            parsed_mol = Chem.MolFromSmiles(smiles) if isinstance(smiles, str) else None
            mol = Chem.AddHs(parsed_mol) if parsed_mol is not None else None
            atom_count = mol.GetNumAtoms() if mol is not None else 0
            atom_counts.append(atom_count)
        max_atoms = max(atom_counts) + 5
        print(f"Auto-calculated maximum number of atoms: {max_atoms}")
    
    print("Extracting and collecting atom features for normalization...")
    all_atom_features = []
    for smiles in tqdm(df["SMILES"], desc="Processing Atom Features"):
        parsed_mol = Chem.MolFromSmiles(smiles) if isinstance(smiles, str) else None
        mol = Chem.AddHs(parsed_mol) if parsed_mol is not None else None
        if mol is not None:
            for atom in mol.GetAtoms():
                all_atom_features.append(extract_atom_features(atom))
    
    # Fit on the intended training partition; never refit on held-out data.
    if not all_atom_features:
        raise ValueError("No valid SMILES were available for atom feature scaling.")
    atom_scaler = StandardScaler()
    atom_scaler.fit(np.array(all_atom_features))
    
    bond_scaler = None
    if use_bond_features:
        all_bond_features = []
        print("Extracting and collecting bond features for normalization...")
        for smiles in tqdm(df["SMILES"], desc="Processing Bond Features"):
            parsed_mol = Chem.MolFromSmiles(smiles) if isinstance(smiles, str) else None
            mol = Chem.AddHs(parsed_mol) if parsed_mol is not None else None
            if mol is not None and mol.GetNumBonds() > 0:
                for bond in mol.GetBonds():
                    all_bond_features.append(extract_bond_features(bond))
        
        if len(all_bond_features) > 0:
            bond_scaler = StandardScaler()
            bond_scaler.fit(np.array(all_bond_features))
    
    graph_builder = MoleculeGraphBuilder(atom_scaler, bond_scaler)
    graph_data_list = []
    error_count = 0
    max_error_display = 20
    
    print("Building molecule graphs from SMILES...")
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Building Molecule Graphs"):
        try:
            molecule_graph = graph_builder.build_graph_from_smiles(
                row["SMILES"],
                max_atoms,
                use_bond_features,
                use_global_features
            )
            if molecule_graph is None:
                continue
            
            if 'y_true' in row:
                molecule_graph['y'] = torch.tensor([row['y_true']], dtype=torch.float)
            else:
                raise ValueError("DataFrame is missing the target column 'y_true'")
            
            soft_labels_list = []
            for col in soft_label_columns:
                if col in row:
                    soft_labels_list.append(row[col])
                else:
                    raise ValueError(f"DataFrame is missing the soft label column '{col}'")
            # Preserve the CSV column order and numeric prediction scale.
            molecule_graph['y_soft'] = torch.tensor(soft_labels_list, dtype=torch.float).view(1, -1)
            
            graph_data_list.append(molecule_graph)
        
        except Exception as e:
            error_count += 1
            if error_count <= max_error_display:
                print(f"Error processing SMILES: {row['SMILES']}, Error message: {str(e)}")
            elif error_count == max_error_display + 1:
                print(f"Exceeded maximum error display count ({max_error_display}), subsequent errors will not be shown...")
    
    print(f"Graph building completed | Total errors: {error_count} | Valid samples: {len(graph_data_list)}")
    
    print(f"Saving features to directory: {output_directory}...")
    
    graph_data_save_path = os.path.join(output_directory, "molecule_graph_data.pt")
    torch.save(graph_data_list, graph_data_save_path)
    
    scalers_save_path = os.path.join(output_directory, "feature_scalers.pkl")
    with open(scalers_save_path, 'wb') as f:
        pickle.dump({
            'atom_scaler': atom_scaler,
            'bond_scaler': bond_scaler,
            'max_atoms': max_atoms
        }, f)
    
    metadata_save_path = os.path.join(output_directory, "graph_metadata.txt")
    with open(metadata_save_path, 'w') as f:
        f.write(f"max_atoms={max_atoms}\n")
        f.write(f"use_bond_features={use_bond_features}\n")
        f.write(f"use_global_features={use_global_features}\n")
        f.write(f"num_valid_samples={len(graph_data_list)}\n")
        f.write(f"soft_label_columns={', '.join(soft_label_columns)}\n")
    
    print("Molecule graph feature extraction completed successfully!")
    return output_directory

if __name__ == "__main__":
    # Paths are relative to the repository root / current working directory.
    input_csv_path = os.path.join("data-set", "input", "with_soft_labels.csv")
    output_feature_directory = os.path.join("data-set", "features", "soft_labels")

    df = pd.read_csv(input_csv_path)

    # Column order must match the teacher-prediction interface.
    soft_label_columns = [
        "temp",
        "elem",
        "mol",
        "SMILES_Kmeans",
        "Bemis_Murcko",
    ]

    feature_dir = extract_molecule_graph_features(
        df=df,
        soft_label_columns=soft_label_columns,
        output_directory=output_feature_directory,
        use_bond_features=True,
        use_global_features=True
    )
    print(f"All features saved to directory: {feature_dir}")
